# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Instale Gymnasium si es necesario:

```bash
pip install gymnasium[toy-text]
```


In [ ]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML
from matplotlib import animation


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [ ]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


### Pregunta 1

¿Cuántos estados y cuántas acciones tiene Taxi-v4?

Explique brevemente por qué Taxi tiene muchos más estados que FrozenLake.

**Respuesta:**

Taxi-v4 tiene **500 estados** y **6 acciones**.

Los 500 estados salen de combinar todas las variables del problema: 25 posiciones del taxi (grid 5×5) × 5 ubicaciones posibles del pasajero (R, G, Y, B o dentro del taxi) × 4 destinos (R, G, Y, B):

$$
25\times5\times4=500
$$

FrozenLake 4×4 solo tiene 16 estados porque el estado es únicamente la posición del agente. En Taxi el estado codifica además dónde está el pasajero y a dónde quiere ir, y el número de estados crece de forma multiplicativa con cada variable adicional.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [ ]:
state, info = env.reset(seed=42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)


### Pregunta 2

En la interacción anterior identifique:

$$
s_t,\quad a_t,\quad r_{t+1},\quad s_{t+1}
$$

¿Qué representa cada elemento?

**Respuesta:**

- $s_t$ (`state`): el estado antes de actuar. Es un entero entre 0 y 499 que codifica la posición del taxi, dónde está el pasajero y cuál es su destino.
- $a_t$ (`action`): la acción ejecutada (0–5: South, North, East, West, Pickup, Dropoff).
- $r_{t+1}$ (`reward`): la recompensa que entrega el ambiente por esa transición. En Taxi es $-1$ por cada paso, $+20$ por dejar al pasajero en su destino y $-10$ por hacer Pickup o Dropoff de forma ilegal.
- $s_{t+1}$ (`next_state`): el nuevo estado al que pasa el ambiente después de ejecutar la acción. Si el taxi choca contra una pared, se queda en la misma posición y el estado puede no cambiar.


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [ ]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


### Pregunta 3

¿Cuántos valores debe aprender el agente en total?

Calcule:

$$
|\mathcal{S}| \times |\mathcal{A}|
$$

**Respuesta:**

$$
|\mathcal{S}|\times|\mathcal{A}|=500\times6=3000
$$

El agente debe aprender **3000 valores** $Q(s,a)$, uno por cada celda de la Q-table (shape `(500, 6)`).


## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [ ]:
def choose_action(Q, state, epsilon, env):
    # 1. decidir si explorar o explotar
    if random.random() < epsilon:
        # Exploración: acción aleatoria
        return env.action_space.sample()

    # 2. Explotación: argmax_a Q(s,a), con desempate aleatorio
    q_values = Q[state]
    max_q = np.max(q_values)
    best_actions = np.flatnonzero(q_values == max_q)
    return int(np.random.choice(best_actions))


## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [ ]:
def update_q(Q, state, action, reward, next_state, alpha, gamma, done=False):
    # Si el episodio terminó (estado terminal) no hay valor futuro
    best_next_q = 0.0 if done else np.max(Q[next_state])

    # Calcular el target: r + gamma * max_a' Q(s', a')
    td_target = reward + gamma * best_next_q

    # Calcular el error TD
    td_error = td_target - Q[state, action]

    # Actualizar Q[state, action]
    Q[state, action] += alpha * td_error


## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [ ]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # TODO 1: escoger acción
            action = choose_action(Q, state, epsilon, env)

            # TODO 2: ejecutar acción en el ambiente
            next_state, reward, terminated, truncated, _ = env.step(action)

            # TODO 3: actualizar Q
            # Solo "terminated" corta el valor futuro; si el episodio se trunca
            # por límite de tiempo el estado siguiente sigue teniendo valor.
            update_q(Q, state, action, reward, next_state, alpha, gamma, done=terminated)

            # TODO 4: actualizar estado y recompensa acumulada
            state = next_state
            total_reward += reward

            # TODO 5: terminar si corresponde
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards


## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [ ]:
Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [ ]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v3 — recompensa promedio ({window} episodios)")
plt.show()


### Pregunta 4

Describa la curva de aprendizaje.

- ¿La recompensa promedio mejora?
- ¿Después de aproximadamente cuántos episodios comienza a estabilizarse?
- ¿El comportamiento observado indica convergencia perfecta o solamente una política razonablemente buena?

**Respuesta** (valores aproximados de una ejecución; cambian un poco con la semilla):

- **¿Mejora?** Sí. La recompensa promedio parte de valores muy negativos (alrededor de $-300$ en la primera ventana, porque el agente casi no completa la tarea y acumula $-1$ por paso y penalizaciones de $-10$) y sube de forma sostenida hasta valores positivos (≈ $+2$ a $+3$).
- **¿Cuándo se estabiliza?** Aproximadamente después de **1500–2000 episodios**. Antes de eso hay una fase de mejora rápida y luego la curva se aplana alrededor de ese valor.
- **¿Convergencia perfecta?** No. Es una política **razonablemente buena**, no óptima. Un episodio óptimo en Taxi da alrededor de $+8$ a $+9$ de recompensa, y el promedio de entrenamiento queda por debajo porque se sigue explorando con $\epsilon=0.1$ (acciones aleatorias que cuestan pasos y a veces penalizaciones). Además la curva sigue siendo ruidosa. La política greedy evaluada sin exploración llega a ≈ $+7.5$.


## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [ ]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [ ]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


Ahora observe el agente entrenado.


In [ ]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


### Pregunta 5

Compare los dos episodios.

- ¿Qué diferencias observa en el comportamiento del taxi?
- ¿El taxi sin entrenamiento logra completar la tarea?
- ¿El agente entrenado evita acciones innecesarias?
- ¿Qué evidencia visual le permite afirmar que el agente aprendió?

**Respuesta:**

- **Diferencias:** el taxi sin entrenamiento (Q-table en ceros) elige siempre entre acciones empatadas al azar, así que deambula, choca contra paredes y hace Pickup/Dropoff en lugares incorrectos. El agente entrenado va directo al pasajero, hace Pickup, se dirige al destino y hace Dropoff.
- **¿El taxi sin entrenamiento completa la tarea?** No, en 50 pasos casi nunca lo logra y su recompensa total es muy negativa (por ejemplo $-185$ en una ejecución, por los $-1$ por paso y los $-10$ por acciones ilegales).
- **¿El agente entrenado evita acciones innecesarias?** En general sí: sigue una ruta corta, sin Pickup/Dropoff ilegales y sin dar vueltas. Puede haber algún paso de más, porque la política no es perfectamente óptima.
- **Evidencia visual:** el comportamiento es dirigido y coherente (recoge, transporta y deja al pasajero), el episodio termina en pocos pasos, y la recompensa total pasa de muy negativa a positiva (≈ $+10$ en una ejecución, con la semilla 7). Nada de eso es posible por azar.


## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [ ]:
state = 123

print("Estado:", state)
print("Q-values:", Q_trained[state])
print("Mejor acción:", np.argmax(Q_trained[state]))


### Pregunta 6

Para el estado seleccionado:

1. ¿Cuál es la acción con mayor valor Q?
2. ¿Qué significa que una acción tenga un valor Q mayor que otra?
3. ¿Por qué no podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata de ejecutar la acción?

**Respuesta** (para `state = 123`; los valores exactos varían con la semilla):

El estado 123 se decodifica como: taxi en la fila 1, columna 1; pasajero en R (esquina superior izquierda); destino B. El taxi debe ir a R: West y luego North, y hacer Pickup.

1. La acción con mayor $Q$ es la **3 (West)**, con un valor cercano a $+3.9$ en nuestra ejecución, mientras que las demás acciones tienen valores negativos (por ejemplo, Pickup ≈ $-7$ y Dropoff ≈ $-5.4$ porque son ilegales en ese estado). Es coherente con la ruta hacia el pasajero.
2. Que $Q(s,a_1) > Q(s,a_2)$ significa que, según lo aprendido, ejecutar $a_1$ en $s$ y luego seguir actuando bien produce un **retorno esperado (descontado) mayor** que ejecutar $a_2$. Por eso la política greedy elige el argmax.
3. $Q(s,a)$ no es la recompensa inmediata porque es el **retorno acumulado esperado**: $r+\gamma\max_{a'}Q(s',a')$ incluye las recompensas futuras. West da $-1$ de recompensa inmediata y aun así tiene el mayor $Q$ porque acerca al taxi al pasajero y, finalmente, al $+20$ de la entrega. Una acción con recompensa inmediata baja puede ser la mejor a largo plazo.


## 13. Experimentación

Modifique **solo uno** de los siguientes hiperparámetros y vuelva a entrenar:

- $\alpha$
- $\gamma$
- $\epsilon$

### Pregunta 7

Compare el nuevo entrenamiento con el original.

Explique cómo el cambio del hiperparámetro afectó:

- velocidad de aprendizaje,
- estabilidad,
- recompensa final,
- comportamiento observado.


In [ ]:
# Experimento: modificamos SOLO alpha (0.1 -> 0.5); gamma=0.95 y epsilon=0.1 igual

Q_exp, rewards_exp = train_q_learning(
    env,
    np.zeros((n_states, n_actions)),
    episodes=5000,
    alpha=0.5,      # <-- único cambio (original: 0.1)
    gamma=0.95,
    epsilon=0.1
)

moving_average_exp = np.convolve(
    rewards_exp,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(moving_average, label="alpha = 0.1 (original)")
plt.plot(moving_average_exp, label="alpha = 0.5 (experimento)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi — efecto de alpha (promedio de {window} episodios)")
plt.legend()
plt.show()

print("Std recompensa (últimos 1000 episodios):")
print("  alpha=0.1:", np.std(rewards[-1000:]))
print("  alpha=0.5:", np.std(rewards_exp[-1000:]))


**Respuesta Pregunta 7** (modifiqué solo $\alpha$: de $0.1$ a $0.5$; valores aproximados de una ejecución):

- **Velocidad de aprendizaje:** mucho mayor con $\alpha=0.5$. La recompensa promedio ya es positiva (≈ $+2$) hacia el episodio 500, mientras que con $\alpha=0.1$ tarda 1500–2000 episodios en llegar a ese nivel. Con un $\alpha$ mayor cada experiencia mueve más la estimación hacia el TD target.
- **Estabilidad:** al terminar ambos entrenamientos tienen una variabilidad similar (desviación ≈ $8$ en los últimos 1000 episodios, dominada por la exploración $\epsilon=0.1$). Un $\alpha$ alto puede ser algo más ruidoso porque cada transición sobrescribe más información anterior, aunque en Taxi (ambiente determinístico) este efecto es pequeño.
- **Recompensa final:** casi igual (≈ $+2$ a $+3$ en entrenamiento, y ≈ $+7.5$ a $+8$ en evaluación greedy), porque ambos alcanzan una política similar.
- **Comportamiento observado:** con $\alpha=0.5$ el taxi aprende antes a recoger y entregar al pasajero. Con $\alpha=0.1$ necesita más episodios, pero el aprendizaje es más suave.

Conclusión: como Taxi es determinístico, un $\alpha$ grande acelera el aprendizaje sin costo serio. En ambientes estocásticos un $\alpha$ alto haría que las estimaciones oscilen. (Otras pruebas: $\gamma=0.5$ dificulta mucho el aprendizaje porque el agente no valora la recompensa lejana de $+20$; $\epsilon=0.5$ deja la recompensa de entrenamiento muy baja por exceso de exploración.)

## Entrega

El notebook debe contener:

1. implementación de `choose_action`;
2. implementación de `update_q`;
3. implementación de `train_q_learning`;
4. curva de aprendizaje;
5. visualización del agente antes y después del entrenamiento;
6. respuestas a las siete preguntas.

No es necesario modificar las funciones auxiliares de visualización.
